# DELTA LAKE

Local instance to manage MatrizActividades

In [1]:
import pandas as pd
import re
import json
import logging
import pymongo
from pymongo.errors import ConnectionFailure

from eerssa.secret import Keys


from deltalake import DeltaTable, write_deltalake
import os
import sys

from pprint import pprint

logging.basicConfig(level=logging.INFO)

Success!!!


In [2]:


# --- MongoDB Connection ---
# It's better to establish the connection once and keep it open for the app's lifetime.
# We will also exit if the connection fails, as the consumer can't do its job without it.
uri = Keys.MONGO_KEY.value
client = None  # Initialize client to None
db_eerssa = None
CurrentCollection = None
ReloadCollection = None


try:
    # Add a timeout to avoid blocking indefinitely
    client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=5000)
    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    db_eerssa = client.eerssa                   # Base de datos EERSSA
    CurrentCollection = db_eerssa.ot_v22        # Coleccion actual
    ReloadCollection  = db_eerssa.ot_reemplazo  # Aqui se cargan OTs repetidas
    logging.info(":::: Conexion exitosa con MongoDB ::::")
    
except ConnectionFailure as e:
    logging.error(f"\n\n ><><> Error de conexion a MongoDB: {e}")
    sys.exit(1) # Exit the script if we can't connect to MongoDB, as it's a critical dependency.


INFO:root::::: Conexion exitosa con MongoDB ::::


In [2]:
## RECARGAR LAS LIBRERIAS DINAMICAMENTE
from importlib import reload
from eerssa import gestionOT as OrdenTrabajo             # Convert from PDF_ot to obj_ot
from eerssa import matrizActividades as Actividades     # process ot.data["actividades"]

In [5]:
reload( OrdenTrabajo )
reload( Actividades  )

<module 'eerssa.matrizActividades' from '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/eerssa/matrizActividades.py'>

## DELTA LAKE

### Descargar todo el 2025

Para iniciar crearemos un DeltaLake de las OTs del 2025

In [ ]:
import re
# Assuming CurrentCollection is a valid pymongo.collection.Collection object
# and is already connected to your database as in consumer.py.

regex_pattern = re.compile("^2025")
query_filter = {"fecha": regex_pattern}

# Use find() to get a cursor that points to all matching documents
cursor = CurrentCollection.find(query_filter)

obj_list = []
for document in cursor:
  ot = OrdenTrabajo.GestionOt.from_dict( document )
  obj_list.append( Actividades.ConvertirOT_a_ActividadesCSV(ot))
  #print(f"Processing document with id_ot: {document.get('id_ot')}")
df = pd.concat(obj_list, ignore_index=True)


In [9]:
bkp = df.copy()

In [12]:
df.drop("uuid", axis=1, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23437 entries, 0 to 23436
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Item           23437 non-null  int64 
 1   Cuenta         23437 non-null  object
 2   Evento         23437 non-null  object
 3   Actividad      23390 non-null  object
 4   Alimentador    8004 non-null   object
 5   Primario       23437 non-null  object
 6   Desconexion    23437 non-null  object
 7   SIG            23437 non-null  object
 8   Tipo           20339 non-null  object
 9   Materiales     23437 non-null  object
 10  Cuadrilla      23437 non-null  object
 11  Dia            23437 non-null  object
 12  Fecha          23437 non-null  object
 13  InicioEvento   23437 non-null  object
 14  FinEvento      23435 non-null  object
 15  Duracion       23437 non-null  int64 
 16  Responsable    23437 non-null  object
 17  Colaboradores  23437 non-null  int64 
 18  HorasExtra     23437 non-n

In [14]:
# DELTA_TABLE_PATH_ON_HOST = "/var/lib/docker/volumes/delta_data/_data/my_first_delta_table"
DELTA_TABLE_PATH_ON_HOST = "./test/deltalake_2025"

In [15]:

write_deltalake(DELTA_TABLE_PATH_ON_HOST, df)

In [3]:
dt = DeltaTable("./test/deltalake_2025")

In [11]:
dt.version()

0

In [13]:
import json
text = "{\"id_ot\":59585}"
json.loads( text )

{'id_ot': 59585}